In [25]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [26]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [27]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [28]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [29]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt = f"""
  Please solve the following task:

  {test_case["task"]}
"""

  messages = []
  add_user_message(messages, prompt)
  output = chat(messages)
  return output

In [30]:
def grade_by_model(test_case, output):
  eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")
  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)


In [31]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the result"""
  output = run_prompt(test_case)

  model_grade = grade_by_model(test_case, output)
  score = model_grade["score"]
  reasoning = model_grade["reasoning"]
  strengths = model_grade["strengths"]
  weaknesses = model_grade["weaknesses"]

  return {
    "output": output,
    "test_case": test_case,
    "score": score,
    "reasoning": reasoning,
    "strengths": strengths,
    "weaknesses": weaknesses
  }

In [ ]:
from statistics import mean

def run_eval(dataset):
  """Loads the dataset and calls run_test_case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)
  
  average_score = mean(result["score"] for result in results)
  print(f"Average score: {average_score}")

  return results



In [33]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)


In [34]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validation Regex\n\nHere's a comprehensive solution with multiple approaches:\n\n## Solution 1: Single Regex Pattern (Recommended)\n\n```regex\n^(?!.*(-{2}|\\\\.{2}|\\\\.-|-.))(?!-)(?!.*-$)[a-z0-9][a-z0-9.-]{1,61}[a-z0-9]$|^[a-z0-9]$\n```\n\n## Solution 2: More Readable with Extended Mode\n\n```regex\n^\n  (?!.*(-{2}|\\\\.{2}|\\\\.-|-.))    # Negative lookahead: no consecutive hyphens, periods, or mixed\n  (?!-)                          # Must not start with hyphen\n  (?!.*-$)                       # Must not end with hyphen\n  [a-z0-9]                       # Start with letter or number\n  [a-z0-9.-]{1,61}              # Middle characters (3-63 total)\n  [a-z0-9]                       # End with letter or number\n  $\n|\n^[a-z0-9]$                       # OR single character (edge case)\n```\n\n## Solution 3: Python Implementation\n\n```python\nimport re\n\ndef validate_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates AWS S